# Phase 2: Feature Engineering & Preprocessing

In this phase, we transform the raw traffic data into features suitable for our Random Forest models.

In [ ]:
import pandas as pd
import numpy as np

def extract_features(df):
    # Convert date and extract components
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['hour'] = df['Date'].dt.hour.fillna(8).astype(int)
    df['day_of_week'] = df['Date'].dt.dayofweek.fillna(0).astype(int)
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    df['is_peak']     = df['hour'].apply(
        lambda h: 1 if h in range(8, 10) or h in range(17, 20) else 0
    )

    # Normalize congestion to 0-1 range
    df['congestion_intensity'] = df['Congestion Level'] / 100.0
    return df

DATA_PATH = "../backend/data/raw/Banglore_traffic_Dataset.csv"
df = pd.read_csv(DATA_PATH)
df_processed = extract_features(df)

print("Engineered Features:")
df_processed[['hour', 'is_peak', 'is_weekend', 'congestion_intensity']].head()

## Travel Time Generation

We use the **Travel Time Index** to estimate base travel times for different transport modes (Car, Bus, Metro).

In [ ]:
base_minutes = df_processed['Travel Time Index'] * 20
df_processed['car_time']   = (base_minutes * (1 + df_processed['congestion_intensity'] * 0.8)).round(1)
df_processed['bus_time']   = (base_minutes * (1 + df_processed['congestion_intensity'] * 1.1)).round(1)
df_processed['metro_time'] = (base_minutes * (1 + df_processed['congestion_intensity'] * 0.2)).round(1)

df_processed[['car_time', 'bus_time', 'metro_time']].describe()